# Git 第3周:进阶技巧

> **学习目标**:掌握 stash、bisect、worktree、hooks 等日常高频工具

---

## Day 15:stash — 暂时保存工作

### stash 的核心用途

你正在 feature 分支写代码,突然需要切到 main 修一个紧急 bug。
但当前修改还没到可以 commit 的程度 —— 这时候用 stash。

```bash
git stash                    # 暂存当前工作区的所有改动
git checkout main            # 切到 main 修 bug
git commit ...               # 提交修复
git checkout feature         # 切回来
git stash pop                # 恢复刚才的工作
```

In [ ]:
# 模拟 stash 栈操作

stash_stack = []
worktree = {"file.txt": "modified-v2"}

print(f"初始状态(有修改但未提交): {worktree}")

# 紧急切分支
stash_stack.append(dict(worktree))
worktree = {"file.txt": "clean-v1"}
print(f"git stash 后(工作区干净了): {worktree}")

# 再 stash 一些改动
stash_stack.append({"file2.txt": "new-file"})
print(f"再次 stash,栈深度: {len(stash_stack)}")

# 恢复最近一次
s = stash_stack.pop()
worktree.update(s)
print(f"git stash pop 后: {worktree}")
print(f"stash 栈剩余: {len(stash_stack)}")

## Day 16:bisect — 二分查找 bug 引入点

### bisect 的适用场景

- 发现一个 bug,确定在 v1.0 时没有,在 v2.0 时出现了
- 两个版本之间有好几百个 commit
- 手动检查每一个 commit 不现实

`git bisect` 用二分查找,在 log2(N) 步内找到引入 bug 的 commit。
比如 1024 个 commit 中,最多 10 步就能定位。

In [ ]:
import math, random

total = 1024
bad_at = random.randint(1, 1023)
good, bad, steps = 0, total - 1, 0

while good < bad - 1:
    mid = (good + bad) // 2
    steps += 1
    if mid >= bad_at:
        bad = mid
    else:
        good = mid

print(f"总共 {total} 个 commit,bug 在第 {bad_at} 个 commit 引入")
print(f"bisect 用 {steps} 步找到了 bug 引入点")
print(f"理论最大步数: log2({total}) = {math.ceil(math.log2(total))}")
print()
print("实际使用:")
print("  git bisect start")
print("  git bisect bad HEAD        # 当前版本有 bug")
print("  git bisect good v1.0       # v1.0 确认没问题")
print("  # Git 自动 checkout 中间 commit,测试后标记 good/bad")
print("  # 也可以自动化: git bisect run python test_bug.py")

## Day 17:worktree — 同时操作多个分支

传统方式:同一时间只能在一个分支上工作。切分支 → IDE 重新索引、Docker 可能需要重建、丢失当前编辑状态。

worktree 让你在不同的目录里同时 checkout 多个分支,互不干扰。

```bash
git worktree add ../path branch-name
git worktree add -b new-branch ../path main
git worktree list
git worktree remove ../path
git worktree prune
```

In [ ]:
import subprocess, os

os.chdir('/home/oa/utils/devops-study')

result = subprocess.run("git worktree list", shell=True, capture_output=True, text=True)
print("当前仓库的 worktree 列表:")
print(result.stdout)
print("典型场景:")
print("  ~/project/           ← 主 worktree,开发 feature")
print("  ~/project-hotfix/    ← worktree,修紧急 bug")
print("  两个目录同时工作,互不干扰!")

## Day 18:blame & log 高级用法

| 命令 | 回答的问题 |
|------|-----------|
| `git blame <file>` | 这行代码是谁写的？什么时候写的？ |
| `git log -S "<code>"` | 这段代码是什么时候加入/删除的？ |
| `git log -G "<regex>"` | 哪些 commit 改了匹配正则的代码？ |
| `git log -- <path>` | 这个文件的完整变更历史？ |
| `git log --since/--until/--author` | 按条件过滤历史？ |

## Day 19:submodule & subtree

| 特性 | submodule | subtree |
|------|-----------|---------|
| 存储方式 | 存指向外部仓库特定 commit 的指针 | 将外部仓库代码完整复制到子目录 |
| clone | 需要额外 `git submodule update --init` | 普通 clone 即可 |
| 更新 | 需要手动更新 submodule 指针 | `git subtree pull` |
| 适用 | 公共库/配置,版本独立 | 嵌入第三方代码 |

## Day 20:Git Hooks

### 常用 hooks

| Hook | 触发时机 | 典型用途 |
|------|----------|----------|
| `pre-commit` | `git commit` 之前 | lint、format、检查 TODO |
| `commit-msg` | 编辑完提交信息后 | 校验 commit message 格式 |
| `pre-push` | push 之前 | 运行测试 |
| `post-checkout` | checkout 后 | 安装依赖 |
| `post-merge` | merge 后 | 清理缓存 |

In [ ]:
# 模拟 pre-commit hook

import re

staged_files = {
    "main.py": "def main():\n    # TODO: add error handling\n    run()",
    "utils.py": "def helper():\n    return 42",
}

print("Pre-commit hook: 检查暂存区文件是否有 TODO/FIXME/HACK...")
violations = []
for fname, content in staged_files.items():
    for i, line in enumerate(content.split('\n'), 1):
        if re.search(r'TODO|FIXME|HACK', line):
            violations.append(f"  {fname}:{i}: {line.strip()}")

if violations:
    print("\nFAILED! 发现以下待办标记:")
    for v in violations:
        print(v)
    print("\n请移除或完善 TODO 后再提交。")
else:
    print("\nPASSED! 所有文件通过检查。")

## Day 21:第3周综合练习

In [ ]:
tools = {
    "stash":    "工作区的'快速保存/恢复',随时中断/恢复",
    "bisect":   "用二分法几秒钟定位 bug 引入点(vs 人工几小时)",
    "worktree": "不切换就能同时操作多分支",
    "blame":    "代码考古学,理解每一行代码的来历",
    "hooks":    "把 lint/test 等检查嵌入到 git 操作中,防止引入问题",
}
print("第3周核心收获:")
for name, desc in tools.items():
    print(f"  {name:10s} → {desc}")

print()
print("这些工具是区分'会用 Git'和'精通 Git'的分水岭。")